# Discovery ciego en `LiquidAI/LFM3-1.7B` + **MATH-500** (Colab L4)

Segundo modelo para el método ciego (el 0.8B de Qwen funcionó; el 4B de Qwen casi no loopea con su sampling oficial).
LFM3 es otra familia (13 capas conv + 5 de atención con ventana + 6 globales), piensa con `<think>` y su sampling oficial de mate
(`T=1.0, top_p=0.95, top_k=20`, **sin** presence penalty) no castiga repetirse.

Mismo pipeline que el 4B: elicit (100 a 8k) → diagnose → screen (24 configs, top 3) → select en dev (32) → freeze → test (137 a 16k) → report.
El edit va en `feed_forward.w2` (la down projection de LFM3).

**Candados:** si el base hace loop duro en menos del 10% del elicit, el paso `elicit` se detiene con `STOP:` (~10 min).
Si ningún finalista le gana al base en dev, se congela `no_patch` y el test corre solo el base.

Resultado final: `MyDrive/wsi_runs/FINALES/discover_math500_lfm3-17b.json`. Al final el runtime se apaga solo.


In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
SAMPLING = "lfm3-math"   # T 1.0, top_p 0.95, top_k 20 (model card, Math & STEM)
TAG = MODEL.split("/")[-1].lower().replace(".", "")
RUN_NAME = f"discover_math500_{TAG}"
REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work"
S = f"{REPO}/scripts/discover_math500_blind.py"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm sympy antlr4-python3-runtime==4.11.1 hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
!pip install -q causal-conv1d || echo "causal-conv1d opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)


In [ ]:
import os, glob, shutil, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
BASE_CACHE = "/content/drive/MyDrive/wsi_runs/_base_cache"
VLLM_CACHE = "/content/drive/MyDrive/wsi_runs/_vllm_cache"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True); os.makedirs(BASE_CACHE, exist_ok=True); os.makedirs(VLLM_CACHE, exist_ok=True); os.makedirs(W, exist_ok=True)
for f in glob.glob(f"{DRIVE}/*"):
    if os.path.isfile(f) and not os.path.exists(f"{W}/{os.path.basename(f)}"):
        shutil.copy2(f, W)
os.environ["WSI_SYNC_DIR"] = DRIVE
os.environ["WSI_BASE_CACHE_DIR"] = BASE_CACHE
os.environ["VLLM_CACHE_ROOT"] = VLLM_CACHE
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("ya hecho en Drive:", sorted(os.listdir(DRIVE)) or "nada")
print("base_cache en Drive:", sorted(os.listdir(BASE_CACHE)) or "vacío")


In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
# Chequeo rápido (segundos): ¿transformers y vLLM entienden LFM3? Si esto falla, no sigas: pégame el error.
import transformers, vllm, importlib
print("vllm", vllm.__version__, "| transformers", transformers.__version__)
from transformers import AutoConfig, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_class_from_dynamic_module
c = AutoConfig.from_pretrained(BASE, trust_remote_code=True)
get_class_from_dynamic_module("modeling_lfm3.Lfm3ForCausalLM", BASE)  # importa el código del modelo
print("transformers OK:", c.model_type, c.num_hidden_layers, "capas")
try:
    from vllm import ModelRegistry
    archs = ModelRegistry.get_supported_archs()
    print("vLLM nativo para Lfm3ForCausalLM:", "Lfm3ForCausalLM" in archs)
except Exception as e:
    print("no pude leer el registro de vLLM:", e)


In [ ]:
import time
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9

def run_step(cmd):
    print("
>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"", b"
").split(b"
")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)} (if it says STOP, that is the loop gate, not a bug)")

run_step(["python", "-u", S, "prepare", "--work", W, "--base-dir", BASE, "--model-repo", MODEL, "--sampling", SAMPLING])
run_step(["python", "-u", S, "elicit", "--work", W])
run_step(["python", "-u", S, "diagnose", "--work", W])
run_step(["python", "-u", S, "screen", "--work", W])
run_step(["python", "-u", S, "select", "--work", W])
run_step(["python", "-u", S, "freeze", "--work", W, "--git-commit", COMMIT])
run_step(["python", "-u", S, "test", "--work", W, "--ck-root", "/content/ck"])
run_step(["python", "-u", S, "report", "--work", W])


In [ ]:
src = f"{W}/results_discover_math500_blind.json"
if not os.path.exists(src):
    raise RuntimeError("No se generó results_discover_math500_blind.json; revisa el error arriba.")
res = json.load(open(src))
res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "discover_math500_lfm3_colab.ipynb",
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)


In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
